# Python OOP Advanced — Expert Interview Guide

Covers: `__dunder__` methods, descriptors, metaclasses, MRO, `__slots__`, dataclasses, ABCs, and more.

> **Key mental model:** In Python, *everything* is an object — including classes themselves. Classes are instances of `type`.

## 1. Class Anatomy: `__new__`, `__init__`, `__del__`

- `__new__`: allocates memory, returns the new instance (rarely overridden)

- `__init__`: initializes the instance (most common)

- `__del__`: called when reference count drops to 0 (unreliable — don't rely on it)

`__new__` runs before `__init__`. If `__new__` returns an instance of the class, `__init__` is called on it.

In [ ]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print(f"[__new__] Creating new instance of {cls.__name__}")
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, name: str):
        # Called every time, even if __new__ returns existing instance
        self.name = name

s1 = Singleton("first")
s2 = Singleton("second")
print(s1 is s2)     # True — same object
print(s1.name)      # second (init ran twice on same object)

# __new__ for immutable type customization
class PositiveInt(int):
    def __new__(cls, value):
        if value <= 0:
            raise ValueError(f"PositiveInt must be > 0, got {value}")
        return super().__new__(cls, value)

p = PositiveInt(5)
print(p, type(p))  # 5 <class '__main__.PositiveInt'>
try:
    PositiveInt(-1)
except ValueError as e:
    print(e)

# Instance vs class vs static methods
class MyClass:
    class_var = 0

    def instance_method(self):        # has access to self (instance)
        return f"instance: {self}"

    @classmethod
    def class_method(cls):            # has access to cls (the class)
        cls.class_var += 1
        return f"class: {cls.__name__}, var={cls.class_var}"

    @staticmethod
    def static_method(x, y):         # no self or cls — just a namespaced function
        return x + y

obj = MyClass()
print(obj.instance_method())
print(MyClass.class_method())
print(MyClass.static_method(3, 4))

> **Interview Insight:** `@classmethod` is used for alternative constructors (`from_json`, `from_dict`). `@staticmethod` is a plain function grouped in the class namespace — no implicit first argument.

## 2. Properties: `@property`, `@setter`, `@deleter`

Properties provide controlled attribute access without breaking the public API.

They are **descriptors** under the hood (discussed in section 6).

In [ ]:
class Temperature:
    def __init__(self, celsius: float = 0.0):
        self._celsius = celsius  # private storage

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float):
        if value < -273.15:
            raise ValueError(f"Temperature below absolute zero: {value}")
        self._celsius = value

    @celsius.deleter
    def celsius(self):
        print("Deleting temperature")
        del self._celsius

    @property
    def fahrenheit(self) -> float:
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value: float):
        self.celsius = (value - 32) * 5/9  # validates through celsius setter

t = Temperature(25)
print(t.celsius)      # 25
print(t.fahrenheit)   # 77.0

t.fahrenheit = 32
print(t.celsius)      # 0.0

try:
    t.celsius = -300
except ValueError as e:
    print(e)

del t.celsius

> **Interview Insight:** `@property` lets you start with a plain attribute and add logic later without changing the API. Classic refactoring: `self.name = name` → add `@property` with validation, zero API breakage.

## 3. Inheritance, MRO & `super()`

Python uses **C3 Linearization** to determine Method Resolution Order (MRO).

`super()` follows the MRO — not necessarily calling the direct parent.

In [ ]:
MRO for D(B, C) where B(A) and C(A):
D → B → C → A → object

In [ ]:
class A:
    def method(self):
        print("A.method")
        return "A"

class B(A):
    def method(self):
        print("B.method")
        result = super().method()  # calls C.method (not A!) due to MRO
        return f"B({result})"

class C(A):
    def method(self):
        print("C.method")
        result = super().method()
        return f"C({result})"

class D(B, C):
    def method(self):
        print("D.method")
        result = super().method()
        return f"D({result})"

d = D()
print(d.method())
print("MRO:", [cls.__name__ for cls in D.__mro__])

# super() with explicit args (Python 2 style, avoid in Python 3)
# super(D, self).method()  # same as super().method()

# Cooperative multiple inheritance — each class calls super()
class LogMixin:
    def method(self):
        print("[LOG] before method")
        result = super().method()
        print("[LOG] after method")
        return result

class TimeMixin:
    def method(self):
        print("[TIME] timing start")
        result = super().method()
        print("[TIME] timing end")
        return result

class Base:
    def method(self):
        print("Base.method executing")
        return "base_result"

class Service(LogMixin, TimeMixin, Base):
    pass

svc = Service()
svc.method()
print("MRO:", [c.__name__ for c in Service.__mro__])

> **Interview Insight:** `super()` in cooperative inheritance doesn't mean 'my parent' — it means 'next in MRO'. All mixins must call `super()` for the chain to work correctly.

## 4. Abstract Base Classes (ABC)

ABCs define a contract that subclasses must implement.

`@abstractmethod` prevents instantiating incomplete subclasses.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        """Return area of the shape."""
        ...

    @abstractmethod
    def perimeter(self) -> float:
        """Return perimeter."""
        ...

    # Concrete method in ABC — shared by all subclasses
    def describe(self) -> str:
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

    @classmethod
    @abstractmethod
    def from_dict(cls, data: dict) -> "Shape":
        """Abstract classmethod for alternate constructor."""
        ...

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        import math
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        import math
        return 2 * math.pi * self.radius

    @classmethod
    def from_dict(cls, data: dict) -> "Circle":
        return cls(data["radius"])

# Try to instantiate ABC
try:
    s = Shape()
except TypeError as e:
    print(f"Cannot instantiate ABC: {e}")

c = Circle(5)
print(c.describe())
print(Circle.from_dict({"radius": 3}).area())

# ABC registration — virtual subclass
from abc import ABCMeta
class Flyable(ABC):
    @abstractmethod
    def fly(self) -> str: ...

class Duck:
    def fly(self) -> str:
        return "Duck flying"

Flyable.register(Duck)  # register without inheritance
print(isinstance(Duck(), Flyable))  # True
print(issubclass(Duck, Flyable))    # True

> **Interview Insight:** `ABC.register()` adds a 'virtual subclass' without inheritance — `isinstance` returns True but the class doesn't inherit methods. Useful for third-party class integration.

## 5. `__dunder__` Methods — The Python Data Model

Dunder methods let your classes integrate with Python's built-in operators and protocols.

| Category | Methods |

|---|---|

| Representation | `__repr__`, `__str__`, `__format__` |

| Comparison | `__eq__`, `__hash__`, `__lt__`, `__le__`, `__gt__`, `__ge__` |

| Container | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__` |

| Numeric | `__add__`, `__mul__`, `__sub__`, `__truediv__`, `__radd__` |

| Context | `__enter__`, `__exit__` |

In [ ]:
from functools import total_ordering

@total_ordering  # generates __le__, __gt__, __ge__ from __eq__ and __lt__
class Version:
    def __init__(self, major: int, minor: int, patch: int):
        self.major = major
        self.minor = minor
        self.patch = patch

    def __repr__(self) -> str:
        return f"Version({self.major}, {self.minor}, {self.patch})"

    def __str__(self) -> str:
        return f"{self.major}.{self.minor}.{self.patch}"

    def __eq__(self, other) -> bool:
        if not isinstance(other, Version): return NotImplemented
        return (self.major, self.minor, self.patch) == (other.major, other.minor, other.patch)

    def __lt__(self, other) -> bool:
        if not isinstance(other, Version): return NotImplemented
        return (self.major, self.minor, self.patch) < (other.major, other.minor, other.patch)

    def __hash__(self) -> int:
        return hash((self.major, self.minor, self.patch))

v1 = Version(1, 2, 3)
v2 = Version(1, 3, 0)
print(v1 < v2)    # True
print(v2 > v1)    # True (via @total_ordering)
print(sorted([v2, v1]))

# Custom container
class CircularBuffer:
    def __init__(self, capacity: int):
        self._buf = [None] * capacity
        self._capacity = capacity
        self._size = 0
        self._head = 0

    def append(self, item):
        idx = (self._head + self._size) % self._capacity
        if self._size < self._capacity:
            self._size += 1
        else:
            self._head = (self._head + 1) % self._capacity
        self._buf[idx] = item

    def __len__(self) -> int: return self._size
    def __getitem__(self, idx): return self._buf[(self._head + idx) % self._capacity]
    def __contains__(self, item): return any(self._buf[i] == item for i in range(self._size))
    def __iter__(self):
        for i in range(self._size):
            yield self[i]
    def __repr__(self): return f"CircularBuffer({list(self)})"

cb = CircularBuffer(3)
for x in [1, 2, 3, 4]:  # 4 overwrites 1
    cb.append(x)
print(cb)           # CircularBuffer([2, 3, 4])
print(3 in cb)      # True
print(len(cb))      # 3

> **Interview Insight:** Return `NotImplemented` (not `NotImplementedError`) from comparison methods when the type is unsupported. This allows Python to try the reflected operation on the other object.

## 6. Descriptors: `__get__`, `__set__`, `__delete__`

Descriptors power `@property`, `@classmethod`, `@staticmethod`, and ORM fields.

- **Non-data descriptor**: only `__get__` — functions, `@staticmethod`, `@classmethod`

- **Data descriptor**: `__get__` + `__set__` (and/or `__delete__`) — `@property`

Data descriptors take priority over instance `__dict__`.

In [ ]:
# Custom descriptor — reusable validator
class Validated:
    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f"_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # accessed from class, not instance
        return getattr(obj, self.private_name, None)

    def __set__(self, obj, value):
        value = self.validate(value)
        setattr(obj, self.private_name, value)

    def validate(self, value):
        return value  # override in subclass

class PositiveNumber(Validated):
    def validate(self, value):
        if not isinstance(value, (int, float)) or value <= 0:
            raise ValueError(f"{self.name} must be a positive number, got {value!r}")
        return value

class NonEmptyString(Validated):
    def validate(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{self.name} must be a non-empty string")
        return value.strip()

class Product:
    name = NonEmptyString()
    price = PositiveNumber()
    quantity = PositiveNumber()

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    def total(self): return self.price * self.quantity

p = Product("Widget", 9.99, 100)
print(p.name, p.price, p.total())

try:
    p.price = -5
except ValueError as e:
    print(e)

# How @property is a descriptor internally
class MyProperty:
    def __init__(self, fget=None, fset=None):
        self.fget = fget
        self.fset = fset

    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return self.fget(obj)

    def __set__(self, obj, value):
        if self.fset is None: raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def setter(self, fset):
        return MyProperty(self.fget, fset)

> **Interview Insight:** `__set_name__` (Python 3.6+) is called when the class is defined, giving the descriptor its attribute name automatically. Without it, you'd have to pass the name manually.

## 7. Metaclasses

A metaclass is the class of a class. `type` is the default metaclass.

In [ ]:
instance → class → metaclass
my_obj   → MyClass → type

Metaclasses let you customize class **creation** — add methods, validate, register subclasses.

In [ ]:
# type is a metaclass — classes are instances of type
print(type(int))        # <class 'type'>
print(type(list))       # <class 'type'>
print(type(type))       # <class 'type'>  -- type is its own metaclass

# Custom metaclass
class RegistryMeta(type):
    _registry: dict = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:  # don't register the base class itself
            mcs._registry[name] = cls
        return cls

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

class Plugin(metaclass=RegistryMeta):
    def run(self): raise NotImplementedError

class AudioPlugin(Plugin):
    def run(self): return "Playing audio"

class VideoPlugin(Plugin):
    def run(self): return "Playing video"

print("Registry:", RegistryMeta._registry)
# Instantiate by name
plugin = RegistryMeta._registry["AudioPlugin"]()
print(plugin.run())

# Singleton via metaclass (thread-safe version)
import threading

class SingletonMeta(type):
    _instances = {}
    _lock = threading.Lock()

    def __call__(cls, *args, **kwargs):
        with cls._lock:
            if cls not in cls._instances:
                instance = super().__call__(*args, **kwargs)
                cls._instances[cls] = instance
        return cls._instances[cls]

class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self, url: str):
        self.url = url

db1 = DatabaseConnection("postgres://localhost/db")
db2 = DatabaseConnection("postgres://other/db")
print(db1 is db2)   # True
print(db1.url)      # postgres://localhost/db (first wins)

> **Interview Insight:** Prefer `__init_subclass__` over metaclasses for most use cases (simpler, more readable). Metaclasses are for framework-level customization — Django models, SQLAlchemy ORM use them.

## 8. `__slots__`

By default, instances store attributes in a `__dict__`. `__slots__` replaces this with a fixed set of slots — saving memory and speeding up attribute access.

| | Normal class | `__slots__` |

|---|---|---|

| Memory (per instance) | ~200+ bytes | ~50-100 bytes |

| Attribute access | dict lookup | direct offset |

| Dynamic attributes | Yes | No |

| Weakref support | Yes | Needs `__weakref__` in slots |

In [ ]:
import sys

class PointNormal:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

class PointSlotted:
    __slots__ = ("x", "y", "z")  # declare allowed attributes

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

pn = PointNormal(1.0, 2.0, 3.0)
ps = PointSlotted(1.0, 2.0, 3.0)

print(f"Normal:  {sys.getsizeof(pn)} bytes + {sys.getsizeof(pn.__dict__)} bytes (dict)")
print(f"Slotted: {sys.getsizeof(ps)} bytes (no dict)")

# Normal class has __dict__, slotted does not
print(hasattr(pn, "__dict__"))  # True
print(hasattr(ps, "__dict__"))  # False

# Cannot add arbitrary attributes to slotted class
try:
    ps.w = 4.0
except AttributeError as e:
    print(f"Cannot add: {e}")

# __slots__ in inheritance
class Base:
    __slots__ = ("x",)

class Child(Base):
    __slots__ = ("y",)  # only add NEW slots, don't repeat 'x'

c = Child()
c.x = 1
c.y = 2
# c.z = 3  # AttributeError

# To allow weakrefs and dynamic attrs, add to slots:
class Flexible:
    __slots__ = ("x", "y", "__dict__", "__weakref__")
    def __init__(self, x, y): self.x = x; self.y = y

f = Flexible(1, 2)
f.extra = "dynamic"  # OK because __dict__ is in slots

> **Interview Insight:** Use `__slots__` when creating millions of small objects (points, events, records). Don't use it for general-purpose classes — it complicates inheritance and removes flexibility.

## 9. Dataclasses

`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__` from field annotations.

In [ ]:
from dataclasses import dataclass, field, KW_ONLY
from typing import ClassVar

@dataclass(order=True, frozen=False)
class Employee:
    # order=True: generates __lt__, __le__, etc. using sort_index
    sort_index: float = field(init=False, repr=False)

    name: str
    department: str
    salary: float
    skills: list[str] = field(default_factory=list)  # mutable default!

    # Class variable — not a field
    company: ClassVar[str] = "TechCorp"

    def __post_init__(self):
        # Runs after __init__ — validation, derived fields
        if self.salary < 0:
            raise ValueError(f"Salary cannot be negative: {self.salary}")
        object.__setattr__(self, "sort_index", self.salary)

    def give_raise(self, amount: float) -> None:
        self.salary += amount
        self.sort_index = self.salary

e1 = Employee("Alice", "Engineering", 90000, ["Python", "Go"])
e2 = Employee("Bob", "Marketing", 70000)

print(e1)
print(e1 < e2)  # False (90k > 70k)
print(sorted([e1, e2]))

# frozen=True makes instances immutable (hashable too)
@dataclass(frozen=True)
class Point:
    x: float
    y: float

    def __add__(self, other: "Point") -> "Point":
        return Point(self.x + other.x, self.y + other.y)

p = Point(1.0, 2.0)
print(p + Point(3.0, 4.0))
print({p})  # hashable — can be in a set

# Dataclass vs NamedTuple vs plain class
from typing import NamedTuple

class Config(NamedTuple):  # immutable, tuple subclass, indexed access
    host: str
    port: int = 8080

c = Config("localhost")
print(c.host, c[1])  # named and indexed access

> **Interview Insight:** Never use `field(default=[])` — always `field(default_factory=list)`. Mutable defaults are shared across instances if you use `default` directly.

## 10. Mixins Pattern

Mixins provide reusable behavior through multiple inheritance without being standalone classes.

In [ ]:
class JsonMixin:
    """Adds JSON serialization to any class with __dict__."""
    def to_json(self) -> str:
        import json
        return json.dumps(self.__dict__, default=str)

    @classmethod
    def from_json(cls, json_str: str):
        import json
        data = json.loads(json_str)
        obj = cls.__new__(cls)
        obj.__dict__.update(data)
        return obj

class ValidateMixin:
    """Adds attribute validation via class-level VALIDATORS dict."""
    VALIDATORS: dict = {}

    def __setattr__(self, name, value):
        if name in self.VALIDATORS:
            value = self.VALIDATORS[name](value)
        super().__setattr__(name, value)

class LogMixin:
    """Logs method calls."""
    def __getattribute__(self, name):
        attr = super().__getattribute__(name)
        if callable(attr) and not name.startswith("_"):
            def logged(*args, **kwargs):
                print(f"[LOG] {self.__class__.__name__}.{name}({args}, {kwargs})")
                return attr(*args, **kwargs)
            return logged
        return attr

# Compose mixins into a class
class User(JsonMixin, ValidateMixin):
    VALIDATORS = {
        "age": lambda v: int(v) if isinstance(v, str) else v,
        "email": lambda v: v.lower().strip(),
    }

    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

u = User("Alice", "30", "ALICE@EXAMPLE.COM")  # age coerced, email lowered
print(u.age, u.email)
json_str = u.to_json()
print(json_str)

restored = User.from_json(json_str)
print(restored.name)

> **Interview Insight:** Mixins should: (1) have `Mixin` in the name, (2) not call `super().__init__()` with specific args, (3) be listed before the base class in MRO: `class MyClass(LogMixin, ValidateMixin, Base)`.